# Практична робота 1

## Формулювання задачі, первинний аудит та аналіз ознак

**Набір даних:** Iris (Ірис) — індивідуальний варіант, **цільова ознака: `sepal width` (ширина чашолистка)**

**Дисципліна:** Машинне навчання

**Виконав(ла):** _(ПІБ студента)_

**Група:** _(вказати групу)_

---

> **Важливо.** Метою цієї практичної роботи є **розуміння та підготовка даних**, а не побудова моделі. Модель у цій роботі не будується — тут формулюється задача, описується набір даних, проводиться його первинний аудит та аналіз числових і категоріальних ознак.

## 1. Формулювання задачі та опис індивідуального набору даних

### 1.1. Назва та джерело набору даних

Використовується класичний набір даних **Iris (Ірис Фішера)**, зібраний Р. Фішером у 1936 р. У цій роботі він завантажується через вбудований завантажувач `sklearn.datasets.load_iris`, який містить точну копію оригінального набору (UCI Machine Learning Repository, "Iris Data Set", https://archive.ics.uci.edu/dataset/53/iris).

### 1.2. Предметна область

Предметна область — **ботаніка / морфометрія рослин**. Набір містить результати вимірювання квіток трьох видів ірису (*Iris setosa*, *Iris versicolor*, *Iris virginica*): довжину та ширину чашолистків (sepal) і пелюсток (petal).

### 1.3. Об'єкти, що описуються рядками таблиці

Кожен рядок таблиці відповідає **одній окремій квітці ірису** (одному ботанічному зразку), для якої виміряно чотири морфометричні ознаки та визначено вид рослини.

### 1.4–1.7. Кількість об'єктів і ознак, цільова змінна, тип задачі, практичний зміст

- **Цільова змінна (за умовою індивідуального варіанта):** `sepal_width` (ширина чашолистка, см) — неперервна кількісна ознака.
- **Тип майбутньої задачі машинного навчання:** **регресія** (прогнозування неперервного числового значення), а не класифікація, як у класичному використанні цього набору даних (де ціллю зазвичай є вид рослини `species`).
- **Ознаки-предиктори:** `sepal_length`, `petal_length`, `petal_width`, а також категоріальна ознака `species` (вид рослини).
- **Практичний зміст майбутнього прогнозування:** у природних умовах виміряти довжину чашолистка, довжину та ширину пелюстки і визначити вид квітки зазвичай простіше й швидше, ніж точно виміряти ширину чашолистка (наприклад, через її нерегулярну форму або пошкодження зразка). Модель, що прогнозує `sepal_width` за іншими ознаками, дозволила б **оцінювати цю характеристику непрямим шляхом**, коли пряме вимірювання ускладнене, відсутнє або дороге.

Кількість об'єктів і ознак наводиться нижче за фактичним результатом `df.shape` (розділ 2).

## 2. Первинний аудит даних

Завантажимо дані та проведемо їх первинний аудит: розмірність, типи стовпців, кількість унікальних значень, пропуски, дублікати та потенційні службові/ідентифікаційні ознаки.

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris

pd.set_option('display.max_columns', None)

# Завантаження даних
iris = load_iris(as_frame=True)
df = iris.frame.copy()

# Перейменування стовпців у зручний вигляд snake_case
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'target']

# Розшифровка кодів виду рослини (0/1/2) у назви видів
species_map = dict(zip(range(3), iris.target_names))
df['species'] = df['target'].map(species_map)
df = df.drop(columns=['target'])

df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


Перші рядки показують структуру таблиці: чотири числові морфометричні ознаки (в сантиметрах) та одну категоріальну ознаку `species` (вид рослини).

In [2]:
df.shape

(150, 5)

**Розмірність таблиці:** `(150, 5)` — набір містить **150 об'єктів (квіток)** та **5 стовпців**: 4 числові ознаки (`sepal_length`, `sepal_width`, `petal_length`, `petal_width`) та 1 категоріальна ознака (`species`). Оскільки цільовою змінною за умовою варіанта є `sepal_width`, реальна кількість ознак-предикторів для майбутньої регресії — **4** (`sepal_length`, `petal_length`, `petal_width`, `species`).

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


In [4]:
df.dtypes

sepal_length    float64
sepal_width     float64
petal_length    float64
petal_width     float64
species             str
dtype: object

**Формальні типи даних:** 4 числові стовпці мають тип `float64`, `species` — текстовий/об'єктний тип. Формальний тип узгоджується зі змістовним: усі вимірювальні ознаки дійсно неперервні, `species` — категорія.

In [5]:
df.nunique()

sepal_length    35
sepal_width     23
petal_length    43
petal_width     22
species          3
dtype: int64

In [6]:
df['species'].value_counts()

species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64

Класи за видом рослини **збалансовані**: по 50 об'єктів кожного з трьох видів.

In [7]:
missing_count = df.isnull().sum()
missing_share = (df.isnull().mean() * 100).round(2)
pd.DataFrame({'кількість_пропусків': missing_count, 'частка_пропусків_%': missing_share})

,кількість_пропусків,частка_пропусків_%
sepal_length,0,0.0
sepal_width,0,0.0
petal_length,0,0.0
petal_width,0,0.0
species,0,0.0


**Пропущені значення відсутні** — 0 у кожному стовпці (0.0%).

In [8]:
duplicate_count = df.duplicated().sum()
print('Кількість повних дублікатів рядків:', duplicate_count)
df[df.duplicated(keep=False)].sort_values(list(df.columns))

Кількість повних дублікатів рядків: 1


,sepal_length,sepal_width,petal_length,petal_width,species
101,5.8,2.7,5.1,1.9,virginica
142,5.8,2.7,5.1,1.9,virginica


Виявлено **1 повний дублікат** — рядки 101 і 142 (`sepal_length=5.8, sepal_width=2.7, petal_length=5.1, petal_width=1.9, species=virginica`). Це варто врахувати на етапі підготовки даних до моделювання (наприклад, при розбитті на train/test).

### Потенційні службові або ідентифікаційні ознаки

- Явного стовпця-ідентифікатора (`id`, `index` тощо) у наборі даних **немає**.
- `species` **не є службовою** ознакою — це змістовна категоріальна ознака (вид рослини), яка в поточному варіанті (регресія, ціль — `sepal_width`) переходить у розряд предиктора.
- Індекс `pandas` (0–149) — суто технічний порядковий номер рядка, не несе змістовної інформації.

## 3. Аналіз числових ознак

**Важливо: числові ознаки визначаються за їхнім змістом, а не лише за типом даних `pandas`.** Формальний тип `float64`/`int64` сам собою не гарантує, що ознака є "справжньою" кількісною величиною — іноді числами кодують категорії. Продемонструємо це на прикладі того ж самого набору даних.

In [9]:
# Демонстрація "пастки": у СИРОМУ наборі sklearn вид рослини закодований числом
raw = load_iris(as_frame=True).frame
print(raw.dtypes)
print()
print("Унікальні значення стовпця 'target':", raw['target'].unique())

sepal length (cm)    float64
sepal width (cm)     float64
petal length (cm)    float64
petal width (cm)     float64
target                 int64
dtype: object

Унікальні значення стовпця 'target': [0 1 2]


У сирому наборі стовпець `target` має тип `int64` — формально це число, тому наївна перевірка "числова = та, що має числовий `dtype`" **помилково зарахувала б цей стовпець до числових**. Насправді значення `0, 1, 2` — це лише **коди категорій (видів рослини)**, а не кількісна величина: обчислювати для них середнє, стандартне відхилення чи медіану не має сенсу (наприклад, "середній вид рослини = 1.0" — беззмістовний результат). Саме тому в цій роботі такий стовпець одразу декодовано в текстову категоріальну ознаку `species` (див. розділ 2) і не включено до числового аналізу.

**Висновок:** за змістом (а не лише за `dtype`) справжніми числовими (неперервними, вимірюваними) ознаками в цьому наборі є рівно чотири стовпці — `sepal_length`, `sepal_width`, `petal_length`, `petal_width` (усі — фізичні виміри в сантиметрах). `species` числовою ознакою не є, незалежно від того, як вона закодована (текстом чи числом).

In [10]:
num_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

stats = df[num_cols].agg(
    ['count', 'mean', 'std', 'min',
     lambda x: x.quantile(0.25), 'median',
     lambda x: x.quantile(0.75), 'max']
).T
stats.columns = ['count', 'mean', 'std', 'min', 'Q1', 'median', 'Q3', 'max']

# Додаткові похідні характеристики для аналізу
stats['mean_minus_median'] = (stats['mean'] - stats['median']).round(3)
stats['range'] = stats['max'] - stats['min']
stats['IQR'] = stats['Q3'] - stats['Q1']
stats['CV_%'] = (stats['std'] / stats['mean'] * 100).round(1)

stats.round(3)

,count,mean,std,min,Q1,median,Q3,max,mean_minus_median,range,IQR,CV_%
sepal_length,150.0,5.843,0.828,4.3,5.1,5.80,6.4,7.9,0.043,3.6,1.3,14.2
sepal_width,150.0,3.057,0.436,2.0,2.8,3.00,3.3,4.4,0.057,2.4,0.5,14.3
petal_length,150.0,3.758,1.765,1.0,1.6,4.35,5.1,6.9,-0.592,5.9,3.5,47.0
petal_width,150.0,1.199,0.762,0.1,0.3,1.30,1.8,2.5,-0.101,2.4,1.5,63.6


**Основні статистичні характеристики числових ознак** (кількість непорожніх значень, середнє, стандартне відхилення, мінімум, Q1, медіана, Q3, максимум) наведено в таблиці вище. Усі чотири ознаки мають повні 150 непорожніх значень (пропусків немає, що узгоджується з розділом 2).

In [11]:
df[num_cols].skew().rename('асиметрія (skew)').to_frame()

,асиметрія (skew)
sepal_length,0.314911
sepal_width,0.318966
petal_length,-0.274884
petal_width,-0.102967


### Різниця між середнім і медіаною

| Ознака | Середнє | Медіана | Середнє − медіана | Асиметрія |
|---|---|---|---|---|
| `sepal_length` | 5.843 | 5.80 | **+0.043** (мала) | +0.315 (легка правостороння) |
| `sepal_width` | 3.057 | 3.00 | **+0.057** (мала) | +0.319 (легка правостороння) |
| `petal_length` | 3.758 | 4.35 | **−0.592** (помітна) | −0.275 (легка лівостороння) |
| `petal_width` | 1.199 | 1.30 | **−0.101** (мала) | −0.103 (майже симетрична) |

- `sepal_length` і `sepal_width` — середнє трохи більше за медіану, розподіл близький до симетричного з дуже легкою правосторонньою асиметрією.
- `petal_length` має найбільшу за модулем різницю "середнє − медіана" (−0.592). Це **не типова асиметрія одного розподілу**, а наслідок того, що ознака фактично є **сумішшю трьох підгруп за видами рослин**: у *setosa* довжина пелюстки дуже мала (~1.4 см), а у *versicolor*/*virginica* — значно більша (~4.3–5.6 см). Через це загальний розподіл має два "згустки" значень, і проста статистика (середнє, медіана) для всієї вибірки описує його лише частково.
- `petal_width` асиметрична незначно менше за `petal_length`, з тієї ж причини (суміш видів), але ефект слабший.

### Діапазон значень (range = max − min)

| Ознака | Min | Max | Діапазон |
|---|---|---|---|
| `sepal_length` | 4.3 | 7.9 | 3.6 см |
| `sepal_width` | 2.0 | 4.4 | 2.4 см |
| `petal_length` | 1.0 | 6.9 | **5.9 см (найбільший)** |
| `petal_width` | 0.1 | 2.5 | 2.4 см |

Найбільший абсолютний діапазон — у `petal_length` (5.9 см), оскільки довжина пелюстки найсильніше відрізняється між видами рослин.

### Ознаки з дуже великим розкидом

Оскільки ознаки виміряні в одних одиницях (см), але мають різний масштаб середніх значень, коректніше порівнювати **відносний** розкид — коефіцієнт варіації `CV = std / mean × 100%`:

| Ознака | Std | CV, % |
|---|---|---|
| `sepal_length` | 0.828 | 14.2% |
| `sepal_width` | 0.436 | 14.3% |
| `petal_length` | 1.765 | **47.0%** |
| `petal_width` | 0.762 | **63.6%** (найбільший) |

`petal_width` і `petal_length` мають у 3–4 рази більший відносний розкид, ніж обидві ознаки чашолистка. **Пояснення:** розміри пелюсток дуже сильно різняться між трьома видами (майже неперетинні "кластери" значень для *setosa* проти *versicolor*/*virginica*), тоді як розміри чашолистка варіюють подібно в межах усіх видів і більше перетинаються. Тобто великий розкид тут — це не помилка чи "шумні" дані, а відображення реальної біологічної різноманітності між видами.

### Підозрілі мінімальні або максимальні значення

In [12]:
print("petal_width == min (0.1), розподіл за видами:")
print(df.loc[df['petal_width'] == df['petal_width'].min(), 'species'].value_counts())
print()
print("petal_width == max (2.5), розподіл за видами:")
print(df.loc[df['petal_width'] == df['petal_width'].max(), 'species'].value_counts())
print()
print("sepal_width — мінімум і максимум:")
print(df.loc[df['sepal_width'].isin([df['sepal_width'].min(), df['sepal_width'].max()]),
             ['sepal_width', 'species']])

petal_width == min (0.1), розподіл за видами:
species
setosa    5
Name: count, dtype: int64

petal_width == max (2.5), розподіл за видами:
species
virginica    3
Name: count, dtype: int64

sepal_width — мінімум і максимум:
    sepal_width     species
15          4.4      setosa
60          2.0  versicolor


In [13]:
# Формальна перевірка на викиди за правилом 1.5*IQR
for c in num_cols:
    q1, q3 = df[c].quantile(0.25), df[c].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = df[(df[c] < lo) | (df[c] > hi)]
    print(f"{c}: межі IQR = [{lo:.2f}, {hi:.2f}], знайдено викидів: {len(outliers)}")
    if len(outliers):
        print(outliers[[c, 'species']].to_string(index=True))
    print()

sepal_length: межі IQR = [3.15, 8.35], знайдено викидів: 0

sepal_width: межі IQR = [2.05, 4.05], знайдено викидів: 4
    sepal_width     species
15          4.4      setosa
32          4.1      setosa
33          4.2      setosa
60          2.0  versicolor

petal_length: межі IQR = [-3.65, 10.35], знайдено викидів: 0

petal_width: межі IQR = [-1.95, 4.05], знайдено викидів: 0



- **`petal_width` = 0.1 см (мінімум)** трапляється **5 разів, і завжди у виду *setosa***. Повторюваність значення в межах однієї видової групи свідчить, що це **реальна біологічна особливість** (у *setosa* дуже вузькі пелюстки), а не поодинока помилка вводу чи викид.
- **`petal_width` = 2.5 см (максимум)** трапляється 3 рази, завжди у *virginica* — так само узгоджується з видом рослини, підозр не викликає.
- **За формальним правилом 1.5×IQR** викиди виявлено лише в ознаці **`sepal_width`** — 4 значення: 4.4, 4.1, 4.2 (усі *setosa*) та 2.0 (*versicolor*). Перевірка приналежності до виду показує, що це не випадкові аномалії, а **природна варіативність усередині конкретних видів** (у *setosa* трапляються екземпляри з незвично широким чашолистком, а серед *versicolor* — з незвично вузьким). Значення фізично правдоподібні (додатні, у розумних межах для квітки), тому як помилки даних їх трактувати немає підстав — але при побудові моделі варто пам'ятати про їх існування (перевірити вплив на регресію, за потреби застосувати робастні методи).
- Для `petal_length` і `petal_width` формальне правило 1.5×IQR **не виявляє жодного викиду** — це очікувано, оскільки їхній `IQR` великий саме через змішування трьох видових підгруп, і межі виходять дуже широкими.
- Явно неправдоподібних значень (від'ємних, нульових, аномально великих) в жодній числовій ознаці не виявлено.

## 4. Аналіз категоріальних ознак

Так само, як і з числовими ознаками, категоріальні ознаки визначаються **за змістом**: навіть якби вид рослини зберігався як число (`0/1/2`, як у сирому наборі `sklearn`), це все одно була б **категоріальна (номінальна)** ознака — номери тут лише позначають клас, а не кількість. У нашому робочому наборі даних є **рівно одна** ознака такого типу — `species`.

In [14]:
print("Кількість унікальних категорій:", df['species'].nunique())
print()
print("Частоти категорій (кількість):")
print(df['species'].value_counts())
print()
print("Частки категорій (%):")
print((df['species'].value_counts(normalize=True) * 100).round(2))
print()
print("Пропущені значення у 'species':", df['species'].isnull().sum())

Кількість унікальних категорій: 3

Частоти категорій (кількість):
species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64

Частки категорій (%):
species
setosa        33.33
versicolor    33.33
virginica     33.33
Name: proportion, dtype: float64

Пропущені значення у 'species': 0


In [15]:
# Перевірка на можливі різні варіанти запису тієї самої категорії
# (регістр, зайві пробіли тощо)
print("Унікальні значення 'як є':", sorted(df['species'].unique()))
print("Кількість унікальних значень 'як є':", df['species'].nunique())
print("Кількість унікальних значень після приведення до нижнього регістру й обрізання пробілів:",
      df['species'].str.strip().str.lower().nunique())

Унікальні значення 'як є': [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
Кількість унікальних значень 'як є': 3
Кількість унікальних значень після приведення до нижнього регістру й обрізання пробілів: 3


### Підсумок аналізу категоріальної ознаки `species`

| Характеристика | Значення |
|---|---|
| Кількість унікальних категорій | 3 (`setosa`, `versicolor`, `virginica`) |
| Частоти категорій | по 50 об'єктів кожна (33.33% / 33.33% / 33.33%) — **ідеально збалансовано** |
| Рідкісні категорії | **відсутні** — усі три класи рівномірно представлені, порогу "рідкісної" категорії (наприклад, <5%) не досягає жодна |
| Пропущені значення | 0 (0.0%) |
| Варіанти запису тієї самої категорії | **не виявлено** — кількість унікальних значень не змінюється після приведення до нижнього регістру й видалення пробілів (3 → 3); написання уніфіковане |
| Природний порядок між категоріями | **відсутній** — це номінальна ознака: видова назва рослини не має вбудованої шкали "менше → більше" (на відміну, наприклад, від "низький/середній/високий") |
| Кардинальність | **дуже низька** (3 категорії на 150 об'єктів) — проблем, типових для високої кардинальності (розрідженість, надлишкова розмірність після кодування), немає |

**Ознаки високої кардинальності** в цьому наборі даних **відсутні** — `species` єдина категоріальна ознака, і вона має лише 3 значення, тому додаткові прийоми боротьби з високою кардинальністю (групування рідкісних категорій, target encoding, hashing) тут не потрібні.

### Придатність категоріальних ознак для майбутньої моделі

- **`species` доцільно використовувати** як предиктор у майбутній регресійній моделі (`sepal_width` за іншими ознаками) — дані чисті: без пропусків, без варіантів написання, без рідкісних категорій, з дуже низькою кардинальністю.
- **Рекомендоване перетворення:** оскільки природний порядок між видами відсутній (ознака номінальна), для лінійних та відстань-орієнтованих моделей коректним є **one-hot encoding** (наприклад, `pd.get_dummies(df['species'], drop_first=True)`), а **не** проста нумерація (`label encoding` у вигляді 0/1/2), яка штучно нав'язала б неіснуючий порядок між видами. Для деревоподібних моделей (дерева рішень, випадковий ліс, бустинг) чутливість до способу кодування нижча, але one-hot усе одно є безпечним універсальним вибором при лише 3 категоріях.
- **Додаткового очищення чи виключення ознаки не потрібно** — рідкісних категорій, пропусків чи неоднозначного написання не виявлено.

## 5. Загальний підсумок

| Етап аналізу | Ключовий результат |
|---|---|
| Розмірність | 150 рядків × 5 стовпців |
| Типи стовпців (за змістом) | 4 числові неперервні ознаки (`sepal_length`, `sepal_width`, `petal_length`, `petal_width`) + 1 номінальна категоріальна (`species`) |
| Пропущені значення | 0 у кожному стовпці (0.0%) |
| Повні дублікати рядків | 1 (рядки 101 та 142) |
| Службові/ID-ознаки | відсутні |
| Асиметрія числових ознак | `sepal_*` — легка правостороння; `petal_*` — легка лівостороння через суміш трьох видів у одному розподілі |
| Відносний розкид (CV) | найбільший у `petal_width` (63.6%) і `petal_length` (47.0%) — через відмінності між видами, не через помилки |
| Викиди за правилом 1.5×IQR | лише в `sepal_width` (4 значення); підтверджено, що це природна видова варіативність |
| Категоріальна ознака `species` | 3 збалансовані класи (по 50), без пропусків, без варіантів написання, без природного порядку, низька кардинальність |

**Виявлені потенційні проблеми, які варто врахувати на етапі підготовки даних до моделювання:**
1. Один повний дублікат рядка — розглянути видалення перед розбиттям на train/test.
2. `species` потребує кодування (one-hot, а не label encoding) через відсутність природного порядку.
3. `sepal_width` містить 4 статистичні викиди за правилом IQR — не є помилками даних, але варто перевірити їхній вплив на майбутню регресійну модель.
4. `petal_length` і `petal_width` мають виражену "двогорбу"/змішану структуру розподілу через відмінності між видами — це варто врахувати як важливий сигнал для моделі (`species` як предиктор, ймовірно, суттєво поясн­юватиме частину варіації в цих ознаках і опосередковано — в цільовій `sepal_width`).
5. Пропущені значення відсутні — додаткової обробки не потрібно.

Ці спостереження буде використано на наступних етапах роботи (підготовка ознак, кодування `species`, розбиття на навчальну/тестову вибірки, побудова моделі регресії для прогнозування `sepal_width`), які виходять за межі поточної практичної роботи.